# Projet – Prétraitement de CIC-DDoS2019
## Étape 0 : téléchargement depuis l'URL officielle et lecture en local (Google Colab)

**Ce que fait ce notebook :**
1. télécharge `CSV-01-12.zip` et `CSV-03-11.zip` depuis le site du CIC vers le disque local de Colab (`/content`) ;
2. lit les CSV **directement dans les zip** (pas de décompression des ~30 Go), par morceaux de 500 000 lignes ;
3. compte la distribution **complète** des labels (pour la question 2) ;
4. garde un échantillon (100 % des classes rares, 5 % des autres) et le sauvegarde en Parquet sur Google Drive.

Les sessions suivantes n'ont besoin que de la **cellule 7** (rechargement du Parquet, quelques secondes).

### Avant de commencer : récupérer le cookie de session
Le serveur répond `403 – Registration required.` sans cookie. Pour l'obtenir :
1. Dans Chrome, aller sur https://www.unb.ca/cic/datasets/ddos-2019.html, remplir le formulaire jusqu'à la page qui propose `CSV-01-12.zip`.
2. Appuyer sur **F12** → onglet **Network** (Réseau).
3. Cliquer sur le lien `CSV-01-12.zip` ; une ligne `download.php?file=...` apparaît dans Network. **Annuler** le téléchargement dans le navigateur.
4. Cliquer sur cette ligne → **Headers** → **Request Headers** → copier **toute** la valeur de la ligne `cookie:` (ex. `PHPSESSID=abc123; autre=xyz`).
5. La coller quand la cellule 3 la demande. Elle n'est pas enregistrée dans le notebook.

### Cellule 1 – Imports, graines, versions, Google Drive, paramètres

In [ ]:
import os, sys, zipfile, getpass
import numpy as np
import pandas as pd
from google.colab import drive

# Reproductibilité
SEED = 42
np.random.seed(SEED)

print('Python :', sys.version.split()[0])
print('pandas :', pd.__version__, '| NumPy :', np.__version__)

drive.mount('/content/drive')

# Emplacements
DOSSIER_LOCAL = '/content/cicddos2019'                       # disque Colab (rapide, effacé en fin de session)
DOSSIER_DRIVE = '/content/drive/MyDrive/CICDDoS2019'         # persistant
PARQUET       = f'{DOSSIER_DRIVE}/echantillon.parquet'       # échantillon final
COMPTES       = f'{DOSSIER_DRIVE}/comptes_labels_complets.csv'  # distribution complète (Q2)
os.makedirs(DOSSIER_LOCAL, exist_ok=True)
os.makedirs(DOSSIER_DRIVE, exist_ok=True)

# Source
BASE_URL = 'https://cicresearch.ca/CICDataset/CICDDoS2019/download.php?file=CSVs%2F'
ARCHIVES = {'01-12': 'CSV-01-12.zip',    # jour 1 (12 janvier 2019)
            '03-11': 'CSV-03-11.zip'}    # jour 2 (11 mars 2019)

# Échantillonnage
TAUX  = 0.05                  # fraction gardée pour les classes très fréquentes
RARES = {'BENIGN', 'WebDDoS'} # classes gardées en totalité
CHUNK = 500_000               # lignes lues à la fois

### Cellule 2 – Espace disque disponible
Les deux zip doivent tenir dans `/content` (colonne *Avail*).

In [ ]:
!df -h /content

### Cellule 3 – Téléchargement depuis l'URL
- Téléchargement dans un fichier `.part`, renommé en `.zip` seulement s'il est complet et valide.
- Si la session ou le cookie coupe le téléchargement : **relancer la cellule**, il reprend où il s'était arrêté (`wget -c`).
- Si le serveur renvoie la page d'erreur au lieu du zip, le fichier est supprimé et le message d'erreur est affiché.

In [ ]:
UA      = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36'
REFERER = 'https://www.unb.ca/cic/datasets/ddos-2019.html'

def est_zip_valide(chemin):
    return os.path.exists(chemin) and zipfile.is_zipfile(chemin)

a_telecharger = [n for n in ARCHIVES.values() if not est_zip_valide(f'{DOSSIER_LOCAL}/{n}')]

if os.path.exists(PARQUET):
    print('Échantillon déjà présent sur Drive : rien à télécharger.')
elif not a_telecharger:
    print('Les deux zip sont déjà présents et valides.')
else:
    COOKIE = getpass.getpass('Valeur complète de la ligne "cookie:" : ')
    for nom in a_telecharger:
        url   = BASE_URL + nom
        final = f'{DOSSIER_LOCAL}/{nom}'
        part  = final + '.part'

        # Un .part qui ne commence pas par "PK" est une page d'erreur : on le jette pour ne pas reprendre dessus
        if os.path.exists(part):
            with open(part, 'rb') as f:
                if f.read(2) != b'PK':
                    os.remove(part)

        print(f'\n>>> {nom}')
        !wget -c -q --show-progress --header="Cookie: {COOKIE}" --header="Referer: {REFERER}" -U "{UA}" -O "{part}" "{url}"

        if zipfile.is_zipfile(part):
            os.rename(part, final)
            print(f'{nom} OK : {os.path.getsize(final)/1e9:.2f} Go')
        else:
            with open(part, 'rb') as f:
                debut = f.read(200)
            if debut[:2] == b'PK':
                print(f'{nom} incomplet ({os.path.getsize(part)/1e9:.2f} Go) : relancer la cellule pour reprendre.')
            else:
                os.remove(part)
                raise RuntimeError(f'{nom} : le serveur a renvoyé {debut!r}. '
                                   'Cookie absent ou expiré : en recopier un nouveau et relancer.')

### Cellule 4 – Contenu des archives (question 1 : fichiers CSV et tailles)

In [ ]:
lignes = []
for jour, nom in ARCHIVES.items():
    chemin = f'{DOSSIER_LOCAL}/{nom}'
    if not est_zip_valide(chemin):
        print(f'{nom} absent : cellule ignorée.'); continue
    with zipfile.ZipFile(chemin) as z:
        for info in z.infolist():
            if info.filename.endswith('.csv') and '__MACOSX' not in info.filename:
                lignes.append({'jour': jour, 'fichier': os.path.basename(info.filename),
                               'taille_Go': round(info.file_size / 1e9, 2)})
tailles = pd.DataFrame(lignes)
display(tailles)
if len(tailles):
    print('Total décompressé :', round(tailles['taille_Go'].sum(), 2), 'Go')

### Cellule 5 – Démonstration mémoire (question 3)
On lit 500 000 lignes brutes d'un CSV, puis on convertit `float64 → float32` et `int64 →` plus petit entier possible.
`pd.to_numeric(..., downcast='integer')` choisit un type qui contient toutes les valeurs : aucun dépassement possible.

In [ ]:
def reduire_types(df):
    for c in df.select_dtypes('float64').columns:
        df[c] = df[c].astype('float32')
    for c in df.select_dtypes('int64').columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    return df

chemin = f'{DOSSIER_LOCAL}/CSV-03-11.zip'
if est_zip_valide(chemin):
    with zipfile.ZipFile(chemin) as z:
        premier_csv = [n for n in z.namelist() if n.endswith('.csv') and '__MACOSX' not in n][0]
        with z.open(premier_csv) as f:
            brut = pd.read_csv(f, nrows=CHUNK, low_memory=False)

    avant = brut.memory_usage(deep=True).sum() / 1e6
    print('Types avant :', brut.dtypes.value_counts().to_dict())
    reduit = reduire_types(brut.copy())
    apres = reduit.memory_usage(deep=True).sum() / 1e6
    print('Types après :', reduit.dtypes.value_counts().to_dict())
    print(f'{premier_csv} – {len(brut):,} lignes : {avant:.1f} Mo -> {apres:.1f} Mo (-{100*(1-apres/avant):.1f} %)')
    del brut, reduit

### Cellule 6 – Lecture en local par morceaux + échantillonnage (une seule fois)
Pour chaque morceau de 500 000 lignes :
- on **compte tous les labels** (distribution réelle du jeu complet, pour la question 2) ;
- on garde **100 %** des lignes `BENIGN` et `WebDDoS`, et **5 %** des autres (`random_state=42`) ;
- on ajoute la colonne `jour_capture` (pour la discussion de la question 13 ; à supprimer avant l'apprentissage) ;
- on réduit les types pour limiter la RAM.

Les noms de colonnes sont laissés **bruts** (avec espaces) : leur nettoyage est l'objet de la question 4.

In [ ]:
if os.path.exists(PARQUET):
    print('Échantillon déjà présent sur Drive : passer à la cellule 7.')
else:
    echantillons, comptes = [], []
    for jour, nom in ARCHIVES.items():
        with zipfile.ZipFile(f'{DOSSIER_LOCAL}/{nom}') as z:
            for fichier in z.namelist():
                if not fichier.endswith('.csv') or '__MACOSX' in fichier:
                    continue
                n_lu = n_garde = 0
                with z.open(fichier) as f:
                    for chunk in pd.read_csv(f, chunksize=CHUNK, low_memory=False):
                        col_label = [c for c in chunk.columns if c.strip() == 'Label'][0]
                        chunk[col_label] = chunk[col_label].astype(str).str.strip()

                        vc = chunk[col_label].value_counts()
                        comptes.append(pd.DataFrame({'jour': jour, 'fichier': os.path.basename(fichier),
                                                     'label': vc.index, 'n': vc.values}))

                        rare  = chunk[col_label].isin(RARES)
                        garde = pd.concat([chunk[rare],
                                           chunk[~rare].sample(frac=TAUX, random_state=SEED)])
                        garde['jour_capture'] = jour
                        echantillons.append(reduire_types(garde))
                        n_lu += len(chunk); n_garde += len(garde)
                print(f'{jour} | {os.path.basename(fichier):30s} lues {n_lu:>11,}  gardées {n_garde:>9,}')

    df = pd.concat(echantillons, ignore_index=True)
    del echantillons

    # Colonnes texte (Flow ID, IP, Timestamp, SimillarHTTP...) : forcer en str,
    # sinon un mélange int/str entre morceaux fait échouer l'écriture Parquet.
    for c in df.select_dtypes('object').columns:
        df[c] = df[c].astype(str)

    df.to_parquet(PARQUET, index=False)

    comptes = (pd.concat(comptes).groupby(['jour', 'label'], as_index=False)['n'].sum()
                 .sort_values('n', ascending=False))
    comptes.to_csv(COMPTES, index=False)

    print('\nÉchantillon sauvegardé :', df.shape, '->', PARQUET)
    print('Comptes complets sauvegardés ->', COMPTES)

### Cellule 7 – Rechargement rapide (à utiliser à chaque nouvelle session)

In [ ]:
df = pd.read_parquet(PARQUET)
comptes = pd.read_csv(COMPTES)
col_label = [c for c in df.columns if c.strip() == 'Label'][0]

print('Échantillon :', df.shape, f'| mémoire : {df.memory_usage(deep=True).sum()/1e6:.1f} Mo')

resume = pd.DataFrame({
    'jeu_complet': comptes.groupby('label')['n'].sum(),
    'echantillon': df[col_label].value_counts(),
}).fillna(0).astype(int).sort_values('jeu_complet', ascending=False)
resume['taux_garde_%'] = (100 * resume['echantillon'] / resume['jeu_complet']).round(2)
display(resume)

### Cellule 8 (optionnelle) – Libérer le disque Colab
Une fois le Parquet sauvegardé sur Drive, les zip ne servent plus.

In [ ]:
for nom in ARCHIVES.values():
    for p in (f'{DOSSIER_LOCAL}/{nom}', f'{DOSSIER_LOCAL}/{nom}.part'):
        if os.path.exists(p):
            os.remove(p); print('Supprimé :', p)